# 08 (v2.1). RAG 의료 QA 챗봇 (v2.1 DB + LLM)

v1.1 `04`와 같은 파이프라인을 **v2.1 의료 DB(문단 459,067개)**에 붙입니다. 02·03·06에서 만든 것을 그대로 가져다 씁니다.

```
질문 ─▶ bge 검색 100개 ─▶ MiniLM 재정렬 ─▶ 상위 5개 (중복 제거) ─▶ 점수 낮으면 거절 ─▶ Qwen2.5-7B (+ 06 문장형 LoRA) ─▶ 답변 + 출처 [번호]
```

| 가져다 쓰는 것 | 드라이브 위치 (`rag_msmarco/health/`) | 만든 노트북 |
|---|---|---|
| 문단 DB, FAISS 인덱스, 평가셋 650 | `corpus.jsonl`, `faiss_bge_flatip.index`, `eval.jsonl` | 02 |
| 검증 200 (거절 기준용) | `rerank/val.jsonl` | 03 |
| 문장형 LoRA 어댑터 | `lora_sentence/adapter/` | 06 |

**평가 (v1.1 04보다 넓음)**: 답 있는 건강 500 + **답 없는 건강 100** + 분야 밖 50

| 시스템 | 문서 | 중복 제거 | 거절 기준 | 생성 모델 |
|---|---|---|---|---|
| B0 | 없음 | - | - | 원본 Qwen |
| RAG 기본 | 상위 5개 | ✗ | ✗ | 원본 Qwen |
| RAG 개선 | 상위 5개 | ✓ | ✓ | 원본 Qwen |
| RAG 개선 + LoRA | 상위 5개 | ✓ | ✓ | 원본 Qwen + 06 문장형 LoRA |

⚠️ 이 노트북의 결과는 **v2.1 평가셋 기준**입니다. v1.1 평가 질문 499개가 v2.1 train(06 LoRA 학습 데이터)에 들어 있으므로, 06 LoRA를 v1.1 평가셋으로 재면 안 됩니다.

**런타임: A100 + High-RAM.** 위에서부터 Run all. 전체 약 40~50분 (DB 불러오기 5~8분, Qwen 다운로드 약 5분, 평가 20~30분).
챗봇 셀은 계속 실행 상태로 링크를 띄워 둡니다. 다 쓰면 그 셀의 **■(중지)**를 누르면 마지막 종료 셀이 이어서 실행됩니다.

결과는 드라이브 `rag_msmarco/health/rag/`와 `RAG/2_results/generation_v21/`에 저장됩니다.

## 1. 설치 + 저장 위치

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu peft gradio
!pip -q uninstall -y torchao    # Colab 기본 torchao가 최신 peft와 충돌해서 제거 (06과 같음)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR    = '/content/drive/MyDrive/rag_msmarco/health'                 # v2.1 DB (02)
OUT_DIR     = f'{SAVE_DIR}/rag'
REPORT_DIR  = '/content/drive/MyDrive/RAG/2_results/generation_v21'       # 결과 요약 사본 (Mac RAG 폴더와 동기화)
LORA_DIR    = f'{SAVE_DIR}/lora_sentence/adapter'                         # 06 문장형 LoRA
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
print('DB 위치:', SAVE_DIR, '/ 결과 저장:', OUT_DIR)
print('06 문장형 LoRA 있음:', os.path.isdir(LORA_DIR))

## 2. 설정

시간이 밀리면 `N_ANS`부터 줄이세요 (예: 200). 답 없음 100개와 분야 밖 50개는 항상 전부 씁니다.

In [ ]:
import re, json, time, random, string
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import torch

SEED       = 42
K          = 100        # 1단계 후보 수 (= 재정렬할 후보 수)
TOP_N      = 5          # LLM에 넣을 문단 수
N_ANS      = 500        # 평가할 '답 있는 건강 질문' 수 (최대 500)
GEN_BATCH  = 16
MAX_NEW    = 256
CHAT_LORA  = True       # 챗봇에서 06 문장형 LoRA를 쓸지 (False면 원본 Qwen)

EMB_MODEL  = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
RERANKER   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
LLM_NAME   = 'Qwen/Qwen2.5-7B-Instruct'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'GPU 런타임이 아닙니다. 런타임 유형을 A100으로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB')

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

## 3. 검색기 (v2.1 DB, bge → MiniLM 재정렬)

06과 같은 방식입니다. DB가 커서(문단 46만, 인덱스 1.4GB) 드라이브에서 불러오는 데 5~8분 걸립니다.
- `dedup=True`: 본문이 같은 문단은 하나만 남깁니다.
- 출처 등급: 공공기관 · 병원/학회 · 상업 매체 · 커뮤니티/생활 · 사전 · 기타

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

t = time.time()
corpus   = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
index    = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
texts    = [c['text'] for c in corpus]
assert index.ntotal == len(corpus), '인덱스와 문단 수가 다릅니다'
print(f'문단 {len(corpus):,} / 평가 {len(eval_set)} / 검증 {len(val_set)} ({time.time() - t:.0f}초)')
print(Counter((q['domain'], q['answerable']) for q in eval_set))

emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)
emb_model.half()
reranker = CrossEncoder(RERANKER, max_length=256, device=DEVICE)
reranker.model.half()

TIERS = {
    '공공기관':   ['medlineplus.gov', 'cdc.gov', 'nih.gov', 'nhs.uk', 'who.int', 'fda.gov', 'cancer.gov'],
    '병원·학회':  ['mayoclinic.org', 'clevelandclinic.org', 'hopkinsmedicine.org', 'aafp.org', 'merckmanuals.com',
                  'kidshealth.org', 'health.harvard.edu', 'heart.org', 'cancer.org', 'familydoctor.org', 'uptodate.com'],
    '상업 매체':  ['webmd.com', 'healthline.com', 'verywell.com', 'medicinenet.com', 'everydayhealth.com', 'drugs.com',
                  'rxlist.com', 'emedicinehealth.com', 'medscape.com', 'medicalnewstoday.com', 'rightdiagnosis.com',
                  'patient.info', 'labtestsonline.org'],
    '커뮤니티·생활': ['healthtap.com', 'sharecare.com', 'livestrong.com', 'babycenter.com', 'psychologytoday.com'],
    '사전':       ['thefreedictionary.com'],
}

def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h

def tier_of(site):
    for tier, sites_ in TIERS.items():
        if any(site == s or site.endswith('.' + s) for s in sites_):
            return tier
    return '기타'

sites = [site_of(c['url']) for c in corpus]
norm_text = lambda t: ' '.join(t.lower().split())

def encode(strings):
    return emb_model.encode([QUERY_PREFIX + s for s in strings], batch_size=128, normalize_embeddings=True,
                            convert_to_numpy=True).astype('float32')

def retrieve_many(queries, dedup=True, top_n=TOP_N):
    # 질문마다 [{pid, score, top_raw, site, tier, url, text}, ...] 상위 top_n개
    _, ids = index.search(encode(queries), K)
    pairs = [(q, texts[p]) for q, cands in zip(queries, ids.tolist()) for p in cands]
    all_scores = reranker.predict(pairs, batch_size=512, show_progress_bar=len(queries) > 300).reshape(len(queries), K)
    out = []
    for cands, scores in zip(ids.tolist(), all_scores):
        docs, seen = [], set()
        for i in np.argsort(-scores):
            p = int(cands[i])
            key = norm_text(texts[p])
            if dedup and key in seen:
                continue
            seen.add(key)
            docs.append({'pid': p, 'score': float(scores[i]), 'top_raw': float(scores.max()), 'site': sites[p],
                         'tier': tier_of(sites[p]), 'url': corpus[p]['url'], 'text': texts[p]})
            if len(docs) == top_n:
                break
        out.append(docs)
    return out

retrieve = lambda q, **kw: retrieve_many([q], **kw)[0]

In [ ]:
# 검색 점검: 질문 3개
for q in ['what causes boils to suddenly appear', 'how long does the flu last', 'what is the normal blood sugar level']:
    t = time.time()
    docs = retrieve(q)
    print(f'\n질문: {q}  ({(time.time() - t) * 1000:.0f} ms)')
    for i, d in enumerate(docs, 1):
        print(f'  [{i}] {d["score"]:6.2f}  {d["tier"]:7s} {d["site"]:35s} {d["text"][:80]}')

## 4. 거절 기준 (검증 질문으로만)

04·06과 같은 규칙: 검증 200개(모두 답 있는 건강 질문)의 재정렬 최고 점수 하위 5%보다 낮으면 LLM을 부르지 않고 거절합니다.
시험 질문은 보지 않습니다. (06에서는 4.92였습니다.)

In [ ]:
val_docs = retrieve_many([q['query'] for q in val_set])
THRESH = float(np.percentile([d[0]['top_raw'] for d in val_docs], 5))
val_r5 = np.mean([bool(set(q['gold_pids']) & {d['pid'] for d in docs}) for q, docs in zip(val_set, val_docs)])
print(f'거절 기준: {THRESH:.2f} / 검증 근거 R@5: {val_r5:.3f}')

## 5. LLM (Qwen2.5-7B-Instruct + 06 문장형 LoRA)

- 프롬프트는 04·05·06과 똑같습니다 (LoRA가 이 형식으로 학습됨).
- LoRA를 붙인 채로 `disable_adapter()` 안에서 생성하면 원본 Qwen과 같습니다. 그래서 모델을 한 번만 불러옵니다.
- 데이터가 영어라서 질문도 **영어**로 해야 검색이 잘 됩니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from contextlib import nullcontext

tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side='left')
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.bfloat16, device_map='cuda')
HAS_LORA = os.path.isdir(LORA_DIR)
if HAS_LORA:
    from peft import PeftModel
    llm = PeftModel.from_pretrained(llm, LORA_DIR)
llm.eval()
print('LLM 불러옴:', LLM_NAME, '+ LoRA' if HAS_LORA else '(LoRA 없음)', f'/ GPU 메모리 {torch.cuda.memory_allocated() / 1e9:.1f}GB')

REFUSAL = "Sorry, I can't answer that from the medical documents I have."

SYS_RAG = f'''You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences.'''

SYS_B0 = f'''You are a medical information assistant.
Answer the question from your own knowledge in 2-4 short sentences.
If the question is not about health or medicine, reply with exactly this sentence: "{REFUSAL}"'''

def rag_messages(q, docs):
    ctx = '\n'.join(f'[{i}] ({d["site"]}) {d["text"]}' for i, d in enumerate(docs, 1))
    return [{'role': 'system', 'content': SYS_RAG},
            {'role': 'user', 'content': f'Documents:\n{ctx}\n\nQuestion: {q}'}]

def b0_messages(q):
    return [{'role': 'system', 'content': SYS_B0}, {'role': 'user', 'content': q}]

@torch.inference_mode()
def generate(msgs_list, lora=False, bs=GEN_BATCH):
    # lora=False면 원본 Qwen, True면 06 문장형 LoRA
    ctx = nullcontext() if (lora and HAS_LORA) else (llm.disable_adapter() if HAS_LORA else nullcontext())
    outs = []
    with ctx:
        for i in range(0, len(msgs_list), bs):
            prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list[i:i + bs]]
            enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
            gen = llm.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, temperature=None, top_p=None, top_k=None,
                               pad_token_id=tok.pad_token_id)
            outs += tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

In [ ]:
# 생성 점검: 답 있음 2 + 답 없음 1 + 분야 밖 1
pick = ([q for q in eval_set if q['domain'] == 'health' and q['answerable']][:2] +
        [q for q in eval_set if q['domain'] == 'health' and not q['answerable']][:1] +
        [q for q in eval_set if q['domain'] != 'health'][:1])
for q in pick:
    t = time.time()
    docs = retrieve(q['query'])
    m = [rag_messages(q['query'], docs)]
    print(f"\n[{q['domain']}, 답 {'있음' if q['answerable'] else '없음'}] {q['query']}  (최고 점수 {docs[0]['top_raw']:.2f})")
    print('  원본 :', generate(m)[0])
    if HAS_LORA:
        print('  LoRA :', generate(m, lora=True)[0])
    print('  정답 :', (q.get('well_formed') or q.get('answers') or ['-'])[0][:200], f'({time.time() - t:.1f}초)')

## 6. 평가: B0 vs RAG 기본 vs RAG 개선 vs RAG 개선 + LoRA

**지표**
- 답 있는 건강 질문: **F1**·**ROUGE-L** (짧은 정답), **ROUGE-L 문장형** (문장형 정답이 있는 질문만), **정답 포함률** (짧은 정답 단어가 답변에 들어 있는 비율, 길게 답해도 안 깎임), **출처 표시율**, **잘못 거절**, **근거 R@5**
- 답 없는 건강 질문: **답 없음 거절률** (높을수록 좋음)
- 분야 밖 질문: **분야 밖 거절률** (높을수록 좋음)
- 평균 답변 단어, 질문당 시간

정답 문장에 오타·잘림이 섞여 있어서 F1·ROUGE-L은 **상대 비교용**입니다.

In [ ]:
def toks(s):
    s = s.lower()
    s = re.sub(r'\[\d+\]', ' ', s)                          # 출처 번호는 채점에서 제외
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref)
    common = sum((Counter(p) & Counter(r)).values())
    if not p or not r or common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(r)
    return 2 * prec * rec / (prec + rec)

def rouge_l(pred, ref):
    p, r = toks(pred), toks(ref)
    if not p or not r:
        return 0.0
    dp = [0] * (len(r) + 1)
    for a in p:
        prev = 0
        for j, b in enumerate(r, 1):
            cur = dp[j]
            dp[j] = prev + 1 if a == b else max(dp[j], dp[j - 1])
            prev = cur
    lcs = dp[-1]
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(r)
    return 2 * prec * rec / (prec + rec)

def recall(pred, ref):
    p, r = Counter(toks(pred)), Counter(toks(ref))
    return sum((p & r).values()) / max(1, sum(r.values()))

best = lambda fn, pred, refs: max(fn(pred, r) for r in refs)
has_cite = lambda a: bool(re.search(r'\[\d\]', a))

In [ ]:
all_ans = [q for q in eval_set if q['domain'] == 'health' and q['answerable']]
test_a = random.Random(SEED).sample(all_ans, min(N_ANS, len(all_ans)))
test_n = [q for q in eval_set if q['domain'] == 'health' and not q['answerable']]
test_o = [q for q in eval_set if q['domain'] != 'health']
qs = test_a + test_n + test_o
queries = [q['query'] for q in qs]
print(f'평가: 답 있는 건강 {len(test_a)} (문장형 정답 있음 {sum(bool(q["well_formed"]) for q in test_a)}) '
      f'+ 답 없음 {len(test_n)} + 분야 밖 {len(test_o)}')

runs = {}

t = time.time()
runs['B0'] = {'answers': generate([b0_messages(q) for q in queries]), 'docs': None}
runs['B0']['sec'] = (time.time() - t) / len(qs)
print(f"B0 끝 ({runs['B0']['sec']:.2f}초/질문)")

t = time.time()
docs_raw = retrieve_many(queries, dedup=False)
sec_raw = (time.time() - t) / len(qs)
t = time.time()
docs_dd = retrieve_many(queries, dedup=True)
sec_dd = (time.time() - t) / len(qs)
print(f'검색 끝 (중복 제거 없음 {sec_raw:.2f}초 / 있음 {sec_dd:.2f}초 per 질문)')

systems = [('RAG 기본', docs_raw, sec_raw, False, False),
           ('RAG 개선', docs_dd, sec_dd, True, False)]
if HAS_LORA:
    systems.append(('RAG 개선 + LoRA', docs_dd, sec_dd, True, True))

for name, docs_all, sec_r, use_thresh, lora in systems:
    t = time.time()
    todo = [i for i, d in enumerate(docs_all) if not (use_thresh and d[0]['top_raw'] < THRESH)]
    answers = [REFUSAL] * len(qs)                         # 관련 문서 없음 → LLM 안 부르고 거절
    for i, a in zip(todo, generate([rag_messages(queries[i], docs_all[i]) for i in todo], lora=lora)):
        answers[i] = a
    runs[name] = {'answers': answers, 'docs': docs_all, 'sec': (time.time() - t) / len(qs) + sec_r,
                  'thresh_refused': len(qs) - len(todo)}
    print(f"{name} 끝 ({runs[name]['sec']:.2f}초/질문, 점수 기준 거절 {runs[name]['thresh_refused']}개)")

In [ ]:
na, nn = len(test_a), len(test_n)
A_idx = range(na)
WF_idx = [i for i in A_idx if qs[i]['well_formed']]
N_idx = range(na, na + nn)
O_idx = range(na + nn, len(qs))

rows = {}
for name, r in runs.items():
    A, D = r['answers'], r['docs']
    answered = [i for i in A_idx if not is_refusal(A[i])]
    rows[name] = {
        'F1':             np.mean([best(f1, A[i], qs[i]['answers']) for i in A_idx]),
        'ROUGE-L':        np.mean([best(rouge_l, A[i], qs[i]['answers']) for i in A_idx]),
        'ROUGE-L 문장형':  np.mean([best(rouge_l, A[i], qs[i]['well_formed']) for i in WF_idx]),
        '정답 포함률':     np.mean([best(recall, A[i], qs[i]['answers']) for i in A_idx]),
        '출처 표시율':     np.mean([has_cite(A[i]) for i in answered]) if (D and answered) else np.nan,
        '잘못 거절':       np.mean([is_refusal(A[i]) for i in A_idx]),
        '근거 R@5':        np.mean([bool(set(qs[i]['gold_pids']) & {d['pid'] for d in D[i]}) for i in A_idx]) if D else np.nan,
        '답 없음 거절률':  np.mean([is_refusal(A[i]) for i in N_idx]),
        '분야 밖 거절률':  np.mean([is_refusal(A[i]) for i in O_idx]),
        '평균 답변 단어':  np.mean([len(A[i].split()) for i in answered]) if answered else np.nan,
        '초/질문':        r['sec'],
    }
table = pd.DataFrame(rows).T.round(3)
display(table)

# 정답 문단이 상위 5개 안에 있을 때 / 없을 때 F1 (검색이 답변에 주는 영향)
hit = [bool(set(qs[i]['gold_pids']) & {d['pid'] for d in docs_dd[i]}) for i in A_idx]
split = {name: {'정답 문단 있음': np.mean([best(f1, r['answers'][i], qs[i]['answers']) for i in A_idx if hit[i]]),
                '정답 문단 없음': np.mean([best(f1, r['answers'][i], qs[i]['answers']) for i in A_idx if not hit[i]])}
         for name, r in runs.items()}
split_table = pd.DataFrame(split).T.round(3)
print(f'정답 문단이 상위 5개 안에 있는 질문: {sum(hit)}/{na}')
display(split_table)

summary = '\n'.join([
    f"# 08 RAG 챗봇 평가 (v2.1) ({time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))} KST)", '',
    f'- DB: v2.1 문단 {len(corpus):,}개 / 검색: bge → {RERANKER.split("/")[-1]} 상위 {TOP_N}개 / LLM: {LLM_NAME}'
    + (' + 06 문장형 LoRA' if HAS_LORA else ''),
    f'- 평가: 답 있는 건강 {na} (문장형 정답 {len(WF_idx)}) + 답 없음 {nn} + 분야 밖 {len(test_o)} / '
    f'거절 기준(검증 하위 5%) = {THRESH:.2f}',
    '- F1·ROUGE-L은 짧은 정답, ROUGE-L 문장형은 문장형 정답과의 단어 겹침 (상대 비교용)', '',
    table.to_markdown(), '',
    f'## 정답 문단 유무별 F1 (중복 제거 검색 기준, 정답 문단 있음 {sum(hit)}/{na})', '',
    split_table.to_markdown(),
])
for d in (OUT_DIR, REPORT_DIR):
    with open(f'{d}/results_summary.md', 'w', encoding='utf-8') as f:
        f.write(summary)
    with open(f'{d}/results.json', 'w') as f:
        json.dump({'thresh': THRESH, 'n_ans': na, 'n_noans': nn, 'n_other': len(test_o),
                   'table': table.to_dict('index'), 'split_f1': split_table.to_dict('index')}, f, indent=2, ensure_ascii=False)
    save_jsonl([{'qid': q['qid'], 'domain': q['domain'], 'answerable': q['answerable'], 'query': q['query'],
                 'gold_answer': (q.get('answers') or [''])[0], 'well_formed': (q.get('well_formed') or [''])[0],
                 'gold_pids': q['gold_pids'],
                 **{name: r['answers'][i] for name, r in runs.items()},
                 'rag_docs': [{k: d[k] for k in ('pid', 'score', 'site', 'tier')} for d in docs_dd[i]]}
                for i, q in enumerate(qs)], f'{d}/eval_answers.jsonl')
print('저장:', OUT_DIR, '/', REPORT_DIR)

In [ ]:
# 답변 비교 예시 (번호를 바꿔 보세요. 0~na-1 답 있음, na~ 답 없음, na+nn~ 분야 밖)
for i in [0, 1, na, na + nn]:
    q = qs[i]
    print(f"\n[{q['domain']}, 답 {'있음' if q['answerable'] else '없음'}] {q['query']}")
    print('  정답          :', (q.get('well_formed') or q.get('answers') or ['-'])[0][:250])
    for name in runs:
        print(f'  {name:14s}:', runs[name]['answers'][i])

## 7. 챗봇 (Gradio)

아래 셀을 실행하면 `https://xxxx.gradio.live` 공유 링크가 나옵니다. 폰이나 팀원 컴퓨터에서도 열립니다.
- 검색은 **RAG 개선** 설정(중복 제거 + 거절 기준)을 씁니다. 생성 모델은 `CHAT_LORA`로 고릅니다 (기본: 06 문장형 LoRA).
- 질문과 답은 드라이브 `health/rag/chat_log.jsonl`에 쌓입니다.
- **다 쓰면 이 셀의 ■(중지)를 누르세요.** 챗봇이 닫히고 마지막 종료 셀이 이어서 실행됩니다.

In [ ]:
import gradio as gr

USE_LORA_CHAT = CHAT_LORA and HAS_LORA
EXAMPLES = ['What are the symptoms and causes of conjunctivitis?',
            'How long does the flu last?',
            'What is a normal blood pressure for adults?',
            'Who won the world cup in 2014?']

def answer_one(q):
    t = time.time()
    docs = retrieve(q, dedup=True)
    if docs[0]['top_raw'] < THRESH:
        ans, used = REFUSAL, False
    else:
        ans, used = generate([rag_messages(q, docs)], lora=USE_LORA_CHAT)[0], True
    return ans, docs, used, time.time() - t

def sources_md(docs, used, sec):
    head = f'**근거 문단 상위 {len(docs)}개** · {sec:.1f}초'
    if not used:
        head += f'\n\n⚠️ 재정렬 최고 점수 {docs[0]["top_raw"]:.2f} < 기준 {THRESH:.2f} → 관련 문서가 없다고 보고 답하지 않았어요.'
    parts = [head]
    for i, d in enumerate(docs, 1):
        parts.append(f'**[{i}] {d["tier"]}** · `{d["site"]}` · 점수 {d["score"]:.2f}\n\n> {d["text"]}\n\n[{d["url"]}]({d["url"]})')
    return '\n\n---\n\n'.join(parts)

def chat(message, history):
    message = (message or '').strip()
    if not message:
        return '', history, ''
    ans, docs, used, sec = answer_one(message)
    with open(f'{OUT_DIR}/chat_log.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps({'time': time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time() + 9 * 3600)),
                            'question': message, 'answer': ans, 'answered': used, 'lora': USE_LORA_CHAT,
                            'docs': [{k: d[k] for k in ('pid', 'score', 'site', 'tier')} for d in docs]},
                           ensure_ascii=False) + '\n')
    history = history + [{'role': 'user', 'content': message}, {'role': 'assistant', 'content': ans}]
    return '', history, sources_md(docs, used, sec)

model_label = 'Qwen2.5-7B + 문장형 LoRA' if USE_LORA_CHAT else 'Qwen2.5-7B'
with gr.Blocks(title='의료 QA 챗봇 (RAG 데모, v2.1)') as demo:
    gr.Markdown(f'## 의료 QA 챗봇 · RAG 데모 (v2.1)\nMS MARCO 건강 질문 문단 46만 개에서 근거를 찾아 답합니다 ({model_label}). '
                '**영어로 질문하세요.**')
    with gr.Row():
        with gr.Column(scale=3):
            try:
                bot = gr.Chatbot(type='messages', height=480, label='대화')   # Gradio 5
            except TypeError:
                bot = gr.Chatbot(height=480, label='대화')                    # Gradio 6부터 messages 형식이 기본
            box = gr.Textbox(placeholder='e.g. What causes migraines?', label='질문', autofocus=True)
            gr.Examples(EXAMPLES, inputs=box)
            clear = gr.Button('대화 지우기')
        with gr.Column(scale=2):
            src = gr.Markdown('질문하면 근거 문단이 여기에 나옵니다.')
    gr.Markdown('<small>⚠️ 데모용입니다. 의료 조언이 아니며, 문서는 2016년 무렵 자료라 최신 정보와 다를 수 있습니다.</small>')
    box.submit(chat, [box, bot], [box, bot, src])
    clear.click(lambda: ([], '질문하면 근거 문단이 여기에 나옵니다.'), None, [bot, src])

demo.queue().launch(share=True, prevent_thread_lock=True)
try:
    while True:                       # ■(중지)를 누를 때까지 챗봇을 열어 둠
        time.sleep(1)
except KeyboardInterrupt:
    demo.close()
    print('챗봇을 닫았습니다. 대화 기록:', f'{OUT_DIR}/chat_log.jsonl')

## 8. 드라이브 저장 마무리 + 런타임 종료

결과 파일은 앞 셀들에서 드라이브(`health/rag/`, `RAG/2_results/generation_v21/`)에 저장됐습니다. 이 셀은 저장을 마무리하고 Colab 런타임을 끕니다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()   # 결과 파일 드라이브 저장 마무리
runtime.unassign()          # 런타임 종료